In [ ]:
import sys
import os

import tensorflow as tf

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('encoder.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('decoder.py'), '..')))

import numpy as np
from sklearn.model_selection import train_test_split
import h5py

from variational_ae import VariationalAutoencoder
from encoder import EncoderBuilder
from decoder import DecoderBuilder

In [ ]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"         # Keeps GPU order consistent
os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # Makes only GPU 0 visible (useful even with 1 GPU)

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevents TF from using all GPU memory at once
    except RuntimeError as e:
        print("Error: ", e)
        exit(-1)

In [ ]:
with h5py.File('Dataset/log_mel_spec_data_dataset.h5', 'r') as h5f:
    log_mel_spec_data_train = h5f['train'][:]
    log_melspec_data_labels = h5f['label'][:]

In [ ]:
log_mel_spec_data_train = log_mel_spec_data_train[..., np.newaxis]

log_mel_spec_x_train, log_mel_spec_x_val = train_test_split(log_mel_spec_data_train, test_size=0.05, random_state=42)

In [ ]:
print("Log Spec Train shape:", log_mel_spec_x_train.shape)
print("Log Spec Validation shape:", log_mel_spec_x_val.shape)

In [ ]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
EPOCHS = 50

In [ ]:
input_shape = log_mel_spec_x_train.shape[1:]
latent_space_dim = 128
decoder_out_filter = 1

In [ ]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 1.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [ ]:
conv_layers_config=[
    {'filters': 512, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 256, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 128, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (2, 1)},
]

In [ ]:
dummy_input = tf.random.normal((1, input_shape[0], input_shape[1], input_shape[2]))

In [ ]:
encoder = EncoderBuilder(latent_space_dim, conv_layers_config)
encoder(dummy_input)
shape_before_bottleneck = encoder.get_shape_before_bottleneck()
encoder_model = encoder.build_graph(input_shape=dummy_input.shape)

In [ ]:
decoder = DecoderBuilder(shape_before_bottleneck, decoder_out_filter, conv_layers_config)
dummy_input = tf.random.normal((1, latent_space_dim))
decoder(dummy_input)
decoder_model = decoder.build_graph(input_shape=dummy_input.shape)

In [ ]:
vae = VariationalAutoencoder(
        recon_weight=recon_weight,
        beta=beta,
        encoder=encoder_model,
        decoder=decoder_model
    )

In [ ]:
dummy_input = tf.random.normal((1, input_shape[0], input_shape[1], input_shape[2]))
vae(dummy_input)

In [ ]:
from tensorflow.keras.optimizers import Adam
vae.compile(optimizer=Adam(learning_rate=0.0001))

In [ ]:
vae.summary()

In [ ]:
vae.fit(
    x=log_mel_spec_x_train,
    y=log_mel_spec_x_train, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(log_mel_spec_x_val, log_mel_spec_x_val), # Validation data for monitoring
    shuffle=True
)

In [ ]:
vae.save("initial_vae_log_mel_spec.keras")